In [0]:
%pip install --upgrade numpy pandas pyarrow db-dtypes google-cloud-bigquery google-auth
dbutils.library.restartPython()

# 00 — Criação das Origens de Dados

Este notebook simula as fontes de dados do **Indicador Criança Alfabetizada** (INEP / Base dos Dados).

São criadas duas origens com padrões distintos de ingestão:
1. **Dados estruturados**: UF, municípios e metas nacionais (padrão API)
2. **Streaming**: Cadastros de alunos ao longo do tempo

> **Nota**: Em produção, os dados seriam obtidos diretamente da plataforma
> [Base dos Dados](https://basedosdados.org/) via biblioteca `basedosdados` + BigQuery.
> Esta simulação usa `spark.createDataFrame()` para compatibilidade com Databricks Serverless.

**Próximo passo**: executar `02_carga_camada_bronze.py`

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
import json
import os

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IngesterDados") \
    .getOrCreate()


In [0]:
FILTRO_ANO_INICIAL = 2023
FILTRO_ANO_FINAL = 2024

## 1. Origem Estruturada: Dimensão UF e Metas Nacionais

Simula resposta de API com as 27 Unidades Federativas e metas anuais do programa
**Compromisso Nacional Criança Alfabetizada** (meta: 100% até 2030).

In [0]:
_sa_json_str = dbutils.secrets.get(scope="tc_02_gcp", key="gcp_sa_json")
chave = json.loads(_sa_json_str)

from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_info(chave)
client = bigquery.Client(credentials=credentials, project="aist-tech-challenge02")

import base64

# _sa_json_str já é a string JSON exata do secret — reutilizada sem re-serialização
sa_b64 = base64.b64encode(_sa_json_str.encode("utf-8")).decode("utf-8")

def read_bq_table(table_name: str):
    return (
        spark.read.format("bigquery")
        .option("table", f"basedosdados.br_inep_avaliacao_alfabetizacao.{table_name}")
        .option("parentProject", "aist-tech-challenge02")
        .option("credentials", sa_b64)
        .load()
    )

In [0]:
print("Iniciando carregamento de uf:")
df_uf = read_bq_table("uf")
print("Iniciando carregamento de municipio:")
df_municipio = read_bq_table("municipio")
print("Iniciando carregamento de dicionario:")
df_dicionario = read_bq_table("dicionario")
print("Iniciando carregamento de alunos:")
df_alunos = read_bq_table("alunos")
print("Iniciando carregamento de meta_alfabetizacao_municipio:")
df_meta_alf_mun = read_bq_table("meta_alfabetizacao_municipio")
print("Iniciando carregamento de meta_alfabetizacao_uf:")
df_meta_alf_uf = read_bq_table("meta_alfabetizacao_uf")
print("Iniciando carregamento de meta_alfabetizacao_brasil:")
df_meta_alf_br = read_bq_table("meta_alfabetizacao_brasil")

In [0]:
df_alunos.printSchema()

## 4. Persistência das Origens no Schema `origens`

In [0]:
_hoje = date.today()
_ano  = _hoje.year
_mes  = _hoje.month

spark.sql("CREATE SCHEMA IF NOT EXISTS origens")
# spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

tabelas_origem = {
    "origens.tc02_uf":          df_uf,
    "origens.tc02_meta_brasil": df_meta_alf_br,
    "origens.tc02_meta_uf":     df_meta_alf_uf,
    "origens.tc02_meta_mun":    df_meta_alf_mun,
    "origens.tc02_alunos":      df_alunos,
    "origens.tc02_municipio":   df_municipio,
    "origens.tc02_dicionario":  df_dicionario,
}

for nome_tabela, df in tabelas_origem.items():
    df_com_meta = (
        df
        .withColumn("_data_criacao_origem", F.current_timestamp())
        .withColumn("_ano_ingestao",        F.lit(_ano).cast(T.ShortType()))
        .withColumn("_mes_ingestao",        F.lit(_mes).cast(T.ByteType()))
    )
    (
        df_com_meta.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("_ano_ingestao", "_mes_ingestao")
        .saveAsTable(nome_tabela)
    )

print("Tabelas de origem criadas com sucesso:")
for nome_tabela in tabelas_origem:
    print(f"  - {nome_tabela} => {spark.read.table(nome_tabela).count()} linhas")

print("\nPróximo passo: executar 02_carga_camada_bronze.py")